# Backbone Parameter Counts

This notebook builds parameter-count tables from `configs/backbones.yaml` using `models/parameter_count.py`. The first table keeps ViT-L variants plus the largest configured ViT variant for each backbone family; the second table shows the detailed component breakdown for every configured ViT-family variant.

The tables are architecture-level encoder counts. For an already-loaded `torch.nn.Module` or adapter, use `count_module_parameters(...)` or `count_adapter_parameters(...)` from the same helper module.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'configs' / 'backbones.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not find configs/backbones.yaml from the current notebook path.')


REPO_ROOT = find_repo_root(Path.cwd())
PARAMETER_COUNT_PATH = REPO_ROOT / 'models' / 'parameter_count.py'
spec = importlib.util.spec_from_file_location('probe4physics_parameter_count', PARAMETER_COUNT_PATH)
parameter_count = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = parameter_count
assert spec.loader is not None
spec.loader.exec_module(parameter_count)

build_vit_parameter_table = parameter_count.build_vit_parameter_table
select_vit_size_comparison_rows = parameter_count.select_vit_size_comparison_rows
format_parameter_count = parameter_count.format_parameter_count

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

rows = build_vit_parameter_table(config_path=REPO_ROOT / 'configs' / 'backbones.yaml')
all_counts = pd.DataFrame(rows).sort_values(['backbone', 'total_parameters', 'variant']).reset_index(drop=True)
all_counts['params'] = all_counts['total_parameters'].map(format_parameter_count)
all_counts['fixed_params'] = all_counts['fixed_parameters'].map(format_parameter_count)

comparison = pd.DataFrame(select_vit_size_comparison_rows(rows))
comparison = comparison.sort_values(['backbone', 'total_parameters', 'variant']).reset_index(drop=True)
comparison['params'] = comparison['total_parameters'].map(format_parameter_count)
comparison['fixed_params'] = comparison['fixed_parameters'].map(format_parameter_count)

## ViT-L and largest configured variants

In [ ]:
summary_columns = [
    'backbone',
    'variant',
    'size_label',
    'model_name',
    'is_default_variant',
    'crop_size',
    'frames_per_clip',
    'depth',
    'embed_dim',
    'num_heads',
    'params',
    'total_parameters',
    'fixed_params',
    'formula',
]

comparison[summary_columns].style.format({'total_parameters': '{:,}'})

,backbone,variant,size_label,model_name,is_default_variant,crop_size,frames_per_clip,depth,embed_dim,num_heads,params,total_parameters,fixed_params,formula
0,jepa_v1,vitl16_224,ViT-L,vit_large,False,224,16,24,1024,16,305.49M,"305,490,944",1.61M,upstream_vit_with_frozen_pos_parameter
1,jepa_v1,vith16_384,ViT-H,vit_huge,True,384,16,32,1280,16,637.55M,"637,546,240",5.90M,upstream_vit_with_frozen_pos_parameter
2,jepa_v2,vitl_256,ViT-L,vit_large,False,256,16,24,1024,16,303.89M,"303,885,312",0.00M,upstream_vit_rope
3,jepa_v2,vitg_384,ViT-g,vit_giant_xformers,True,384,16,40,1408,22,1012.17M,"1,012,173,952",0.00M,upstream_vit_rope
4,jepa_v2_1,vitl_384,ViT-L,vit_large,False,384,16,24,1024,16,304.68M,"304,680,960",0.00M,upstream_vit_rope_hierarchical
5,jepa_v2_1,vitG_384,ViT-G,vit_gigantic_xformers,True,384,16,48,1664,26,1845.22M,"1,845,216,768",0.00M,upstream_vit_rope_hierarchical
6,videomae,vit_large_16_224,ViT-L,vit_large,False,224,16,24,1024,16,303.86M,"303,860,736",0.00M,videomae_encoder_formula_qv_bias
7,videomae,vit_huge_16_224,ViT-H,vit_huge,True,224,16,32,1280,16,631.61M,"631,607,040",0.00M,videomae_encoder_formula_qv_bias
8,videomae_v2,vit_large_16_224,ViT-L,vit_large,False,224,16,24,1024,16,303.86M,"303,860,736",0.00M,videomae_encoder_formula_qv_bias
9,videomae_v2,vit_giant_16_224,ViT-g,vit_giant,True,224,16,40,1408,16,1012.12M,"1,012,117,632",0.00M,videomae_encoder_formula_qv_bias


## Detailed component breakdown

In [ ]:
breakdown_columns = [
    'backbone',
    'variant',
    'size_label',
    'model_name',
    'patch_embed_params',
    'patch_embed_img_params',
    'position_embedding_params',
    'blocks_attention_params',
    'blocks_mlp_params',
    'blocks_norm_params',
    'blocks_total_params',
    'final_norm_params',
    'hierarchical_norm_params',
    'modality_embedding_params',
    'total_parameters',
]

all_counts[breakdown_columns].style.format({column: '{:,}' for column in breakdown_columns if column.endswith('_params') or column == 'total_parameters'})

,backbone,variant,size_label,model_name,patch_embed_params,patch_embed_img_params,position_embedding_params,blocks_attention_params,blocks_mlp_params,blocks_norm_params,blocks_total_params,final_norm_params,hierarchical_norm_params,modality_embedding_params,total_parameters
0,jepa_v1,vitl16_224,ViT-L,vit_large,"1,573,888",0,"1,605,632","100,761,600","201,449,472","98,304","302,309,376","2,048",0,0,"305,490,944"
1,jepa_v1,vith16_224,ViT-H,vit_huge,"1,967,360",0,"2,007,040","209,879,040","419,635,200","163,840","629,678,080","2,560",0,0,"633,655,040"
2,jepa_v1,vith16_384,ViT-H,vit_huge,"1,967,360",0,"5,898,240","209,879,040","419,635,200","163,840","629,678,080","2,560",0,0,"637,546,240"
3,jepa_v2,vitl_256,ViT-L,vit_large,"1,573,888",0,0,"100,761,600","201,449,472","98,304","302,309,376","2,048",0,0,"303,885,312"
4,jepa_v2,vith_256,ViT-H,vit_huge,"1,967,360",0,0,"209,879,040","419,635,200","163,840","629,678,080","2,560",0,0,"631,648,000"
5,jepa_v2,vitg_256,ViT-g,vit_giant_xformers,"2,164,096",0,0,"317,419,520","692,362,240","225,280","1,010,007,040","2,816",0,0,"1,012,173,952"
6,jepa_v2,vitg_384,ViT-g,vit_giant_xformers,"2,164,096",0,0,"317,419,520","692,362,240","225,280","1,010,007,040","2,816",0,0,"1,012,173,952"
7,jepa_v2_1,vitb_384,ViT-B,vit_base,"1,180,416","590,592",0,"28,348,416","56,669,184","36,864","85,054,464",0,"6,144","1,536","86,833,152"
8,jepa_v2_1,vitl_384,ViT-L,vit_large,"1,573,888","787,456",0,"100,761,600","201,449,472","98,304","302,309,376",0,"8,192","2,048","304,680,960"
9,jepa_v2_1,vitg_384,ViT-g,vit_giant_xformers,"2,164,096","1,082,752",0,"317,419,520","692,362,240","225,280","1,010,007,040",0,"11,264","2,816","1,013,267,968"
